# Tesla (TSLA) Stock Price Forecasting with Facebook Prophet

This notebook downloads Tesla historical market data directly from Yahoo Finance using `yfinance`, explores the closing-price history, fits a Prophet time-series model, and generates a 30-calendar-day forecast.

> **Important:** This is an educational forecasting demonstration, not investment advice. Forecasts are uncertain and should not be used alone to make financial decisions.

## 1. Install dependencies

Run this cell if the required packages are not already installed.

In [ ]:
%pip install -q yfinance prophet plotly pandas matplotlib

## 2. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go
from prophet import Prophet
from pathlib import Path

pd.set_option("display.max_columns", 30)
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

## 3. Download Tesla historical stock data

The original CSV referenced by the supplied notebook was not available. This version replaces that local-file dependency with an online download of TSLA data. You can change `START_DATE` if you want a shorter history.

In [ ]:
TICKER = "TSLA"
START_DATE = "2010-06-29"

raw = yf.download(
    TICKER,
    start=START_DATE,
    auto_adjust=False,
    progress=False
)

if raw.empty:
    raise RuntimeError("No market data was returned. Check your internet connection or try again later.")

# yfinance may return either flat columns or a MultiIndex, depending on its version.
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

df = raw.reset_index()
df = df.rename(columns={"Date": "Date", "Close": "Price", "Volume": "Volume"})

required = {"Date", "Price"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Downloaded data is missing required columns: {sorted(missing)}")

df["Date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None)
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df = df.dropna(subset=["Date", "Price"]).sort_values("Date").reset_index(drop=True)

df.to_csv(DATA_DIR / "TSLA_historical_data.csv", index=False)
df.head()

## 4. Explore the dataset

In [ ]:
print("Rows and columns:", df.shape)
df.info()
display(df.describe(include="all"))

## 5. Visualize historical performance

In [ ]:
fig = px.line(df, x="Date", y="Price", title="Tesla (TSLA) Historical Closing Price")
fig.update_layout(xaxis_title="Date", yaxis_title="Closing price (USD)")
fig.show()

In [ ]:
if "Volume" in df.columns:
    fig = px.area(df, x="Date", y="Volume", title="Tesla Trading Volume")
    fig.update_layout(xaxis_title="Date", yaxis_title="Volume")
    fig.show()

In [ ]:
fig = px.box(df, y="Price", title="Distribution of Tesla Closing Prices")
fig.show()

## 6. Prepare data for Prophet

Prophet expects two columns: `ds` for dates and `y` for the numeric target. This notebook forecasts the daily closing price.

In [ ]:
prophet_df = df[["Date", "Price"]].rename(columns={"Date": "ds", "Price": "y"})
prophet_df = prophet_df.dropna().sort_values("ds")
prophet_df.head()

## 7. Train the Facebook Prophet model

In [ ]:
model = Prophet()
model.fit(prophet_df)

## 8. Generate a 30-day forecast

The original notebook requested 30 future periods. Prophet's default future frequency is daily, so this creates 30 calendar days beyond the final observation.

In [ ]:
future = model.make_future_dataframe(periods=30, freq="D")
forecast = model.predict(future)
forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(35)

## 9. Visualize forecast and trend components

In [ ]:
forecast_fig = go.Figure()
forecast_fig.add_trace(go.Scatter(x=prophet_df["ds"], y=prophet_df["y"], mode="lines", name="Historical close"))
forecast_fig.add_trace(go.Scatter(x=forecast["ds"], y=forecast["yhat"], mode="lines", name="Prophet forecast"))
forecast_fig.add_trace(go.Scatter(
    x=forecast["ds"], y=forecast["yhat_upper"],
    mode="lines", line=dict(width=0), showlegend=False, name="Upper interval"
))
forecast_fig.add_trace(go.Scatter(
    x=forecast["ds"], y=forecast["yhat_lower"],
    mode="lines", line=dict(width=0), fill="tonexty",
    fillcolor="rgba(99,110,250,0.15)", name="Uncertainty interval"
))
forecast_fig.update_layout(
    title="Tesla Stock Price: Historical Data and 30-Day Prophet Forecast",
    xaxis_title="Date", yaxis_title="Price (USD)"
)
forecast_fig.show()

In [ ]:
model.plot(forecast)
model.plot_components(forecast)

## 10. Export forecast results

In [ ]:
forecast_file = OUTPUT_DIR / "TSLA_30_day_forecast.csv"
forecast.to_csv(forecast_file, index=False)
print(f"Forecast saved to: {forecast_file.resolve()}")

## 11. Notes and limitations

- The original project notebook used a local CSV file that was not supplied with the upload. This cleaned version downloads the data at runtime instead.
- Yahoo Finance data availability, adjusted prices, and historical values may change over time.
- Prophet forecasts daily calendar dates, including dates when the stock market is closed; those dates are model estimates, not actual trading sessions.
- This notebook does not perform a train/test split or report out-of-sample accuracy metrics. Add a proper backtesting workflow before making claims about predictive performance.
- Stock prices are affected by many unpredictable events. This model is for learning and experimentation, not financial advice.